# Test Ollama — AI-ML Playground

Verifies end-to-end connectivity between JupyterHub and the in-cluster Ollama service.

**Requires:** Ollama running in the cluster (`components/ollama/playbook.md`)

**Model:** llama3.2 (default) — served at `http://ollama.ollama:11434`

## STEP 1 — Install packages

In [ ]:
!pip install langchain-ollama langchain-core -q

## STEP 2 — Load playground_config

In [ ]:
import urllib.request, os, sys

config_url = "https://raw.githubusercontent.com/suvmaha/ai-ml-unified-playground-platform/main/components/ollama/playground_config.py"
config_path = os.path.expanduser("~/playground_config.py")

if not os.path.exists(config_path):
    urllib.request.urlretrieve(config_url, config_path)
    print(f"Downloaded playground_config.py → {config_path}")
else:
    print(f"playground_config.py already present at {config_path}")

sys.path.insert(0, os.path.expanduser("~"))
from playground_config import get_llm, get_embeddings, status

status()

## STEP 3 — Verify Ollama is reachable

In [ ]:
import urllib.request

try:
    response = urllib.request.urlopen("http://ollama.ollama:11434/", timeout=5)
    print("✅", response.read().decode())
except Exception as e:
    print("❌ Cannot reach Ollama:", e)
    print("   Is Ollama installed? Run: components/ollama/install.sh")

## STEP 4 — Run inference with llama3.2

In [ ]:
llm = get_llm()  # defaults to ollama + llama3.2

response = llm.invoke("In one sentence: what is Kubernetes?")
print(response.content)

## STEP 5 — Test embeddings (nomic-embed-text)

> First use pulls ~274MB. Allow 1-2 min.

In [ ]:
embeddings = get_embeddings()  # defaults to nomic-embed-text via Ollama

vector = embeddings.embed_query("What is Kubernetes?")
print(f"✅ Embedding dimensions: {len(vector)}")
print(f"   First 5 values: {vector[:5]}")

## STEP 6 — Switch backends (optional)

Same `get_llm()` call works for any backend — no notebook changes needed.

In [ ]:
# Uncomment to try a different backend:

# llm = get_llm(backend="anthropic")  # requires ANTHROPIC_API_KEY
# llm = get_llm(backend="bedrock")    # requires IRSA role on pod
# llm = get_llm(backend="openai")     # requires OPENAI_API_KEY

# response = llm.invoke("In one sentence: what is Ray Serve?")
# print(response.content)